# Strata × Qwen3.8-Flash-Next · Colab 可检查版

目标：原版 Qwen3.8-Flash-Next 的 Strata 量化部署，256K = **262144 个 token，输入、模板、历史、推理及输出共用**。不是把完整 BF16 权重塞入显存。

**先读这段**
1. Colab 菜单「运行时 → 更改运行时类型」选择 GPU；高 RAM 与 GPU 型号/Compute Units 是不同资源，订阅不保证分到指定硬件
2. 依次运行配置与检查。默认只检查，不安装、不下载模型、不启动、不跑推理；所有耗时动作都有独立勾选
3. 检查通过后开启「获取源码」，再开启「安装和下载」。首次约 66–111 GB 模型，加打包、MTP、环境需要更多磁盘，可能很久
4. 开启「应用配置并启动」，先跑短聊天。256K 能启动 ≠ 已验证完整长上下文；最后的长测单独选择
5. 用完停止进程，并在 Colab 主动断开/删除运行时以结束 GPU 资源占用。停止 Strata 不等于停止 Colab 计费

这里没有 Mac 安装步骤，默认不开外部入口。127.0.0.1 是 **Colab 虚拟机内部**；如需本地客户端访问，完成第11–13节的可选认证HTTPS入口。没有保活或规避Colab限制。

实现依据：固定 [Strata v0.1.40.3 / commit d5ea713](https://github.com/Niko1221/Strata/tree/d5ea7133741e67743c0e886bb426c0ce8d69cf6c)、[AI_SETUP](https://github.com/Niko1221/Strata/blob/d5ea7133741e67743c0e886bb426c0ce8d69cf6c/docs/AI_SETUP.md)、[DETAILS](https://github.com/Niko1221/Strata/blob/d5ea7133741e67743c0e886bb426c0ce8d69cf6c/docs/DETAILS.md)。2026-10-08 更新到固定 v0.1.40.3；此版的运行验证见 artifacts/strata-colab/20261008。256K 配置与短聊天验证不等于完整长上下文压力测试。


In [ ]:
#@title 1 · 配置（改完需重跑检查；预设仅建议，不伪造硬件）
preset = "手动256K" #@param ["手动256K", "T4高RAM", "L4", "A10040", "A10080", "G4", "H100"]
apply_preset = False #@param {type:"boolean"}
model_quant = "IQ2_XS" #@param ["Q2_0", "IQ2_XS", "IQ3_XXS", "IQ3_S", "UD-IQ4_XS", "UD-Q4_K_XL"]
context_tokens = 262144 #@param {type:"integer"}
kv_precision = "int8" #@param ["int8", "q4_0", "k8v4"]
kv_streaming = "auto" #@param ["auto", "on", "off"]
cuda_version = "13" #@param ["13", "12"]
accept_experimental_cuda12 = False #@param {type:"boolean"}
low_ram = "auto" #@param ["auto", "on", "off", "resident", "mmap"]
resident_budget_gib = 0 #@param {type:"number"}
expert_cache = "auto" #@param {type:"string"}
vram_reserve_mib = 1024 #@param {type:"integer"}
cpu_pool_workers = 0 #@param {type:"integer"}
prefill_chunk = "auto" #@param ["auto", "256", "512", "1024", "2048", "4096", "8192", "auto:16384", "auto:32768"]
parallel_requests = 1 #@param {type:"integer"}
spec_window = 4 #@param {type:"integer"}
draft_vocab = "cjk" #@param ["cjk", "en", "cyrillic", "fr"]
vision = "no" #@param ["no", "gpu", "cpu"]
vision_tokens = 1024 #@param {type:"integer"}
api_host = "127.0.0.1" #@param ["127.0.0.1", "0.0.0.0"]
api_port = 8080 #@param {type:"integer"}
accept_low_ram_risk = False #@param {type:"boolean"}
accept_experimental_q4 = False #@param {type:"boolean"}
# 仅当 apply_preset=True 时覆盖下面五项；实际 GPU/RAM 仍逐项检查
PRESETS = {
    "T4高RAM": ("Q2_0", 65536, "q4_0", "auto", 1024),
    "L4": ("IQ2_XS", 131072, "int8", "auto", 1024),
    "A10040": ("IQ2_XS", 262144, "int8", "auto", 1536),
    "A10080": ("IQ3_S", 262144, "int8", "auto", 2048),
    "G4": ("IQ2_XS", 131072, "int8", "auto", 1536),
    "H100": ("IQ3_S", 262144, "int8", "auto", 2048),
}
if apply_preset and preset in PRESETS:
    model_quant, context_tokens, kv_precision, kv_streaming, vram_reserve_mib = PRESETS[preset]
print("配置:", model_quant, context_tokens, kv_precision, kv_streaming, "CUDA", cuda_version)


参数说明：
- T4/L4 是低成本起点；T4 的「高 RAM」必须实际验证。G4 是资源标签，Notebook 不把它当作某个确定 GPU，始终读取 nvidia-smi
- 原版四档 2–3.5 位；UD-IQ4_XS/UD-Q4_K_XL 是同一基础模型的 Unsloth 量化，后者实验性。不会自动换成 Coder 或 Swift
- int8 是上游的 8-bit KV 名称，不是 q8_0。q4_0 更省内存，但长文精度可能下降；k8v4 不支持 KV streaming，错误组合会阻止继续
- expert_cache=auto 推荐；整数是 GPU 缓存的专家条目数，不是 GiB。不接受 0。CPU 线程=0 保留 setup 推荐；prefill_chunk 是提示词分块，不是并发；parallel_requests 每加一槽都会增加内存
- resident_budget_gib=0 使用上游默认，仅 Unsloth 支持手动预算。low_ram 可牺牲 I/O 速度降低内存压力，不保证低 RAM 能跑
- spec_window 是 MTP 推测窗口，仅支持 2–8，保留默认 4 最稳；服务模式不能关闭 MTP，权重仍会下载/加载。vision 增加下载、RAM/显存需求
- CUDA 13 要 NVIDIA driver 580+。CUDA 12 是上游实验路径，Linux 驱动最低系列 525（还需满足小版本/构建要求）；不在 Colab 安装或更换宿主驱动


In [ ]:
#@title 2 · 初始化安全执行工具（仅定义函数，不下载）
import os, sys, re, json, time, signal, socket, shutil, secrets, hashlib, platform, subprocess
from pathlib import Path
from urllib.request import Request, urlopen
BASE = Path('/content/strata-colab')
BASE.mkdir(parents=True, exist_ok=True)
REPO, DATA = BASE/'Strata', BASE/'data'
DATA.mkdir(exist_ok=True)
MODELS = DATA/'models'
PIN = 'd5ea7133741e67743c0e886bb426c0ce8d69cf6c'
ORIGIN = 'https://github.com/Niko1221/Strata.git'
STATE = BASE/'owned-server.json'
PROCESSES = globals().get('PROCESSES', {})
# 保留重跑此单元前的密钥；内核重启后生成新密钥，需要停止旧服务再启动
API_KEY = globals().get('API_KEY') or secrets.token_urlsafe(32)
META = {'Q2_0':(66.4,48,34.0), 'IQ2_XS':(68.0,48,35.5), 'IQ3_XXS':(75.8,60,42.9),
        'IQ3_S':(83.6,62,50.3), 'UD-IQ4_XS':(93.7,48,59.5), 'UD-Q4_K_XL':(111.3,48,77.0)}

def validate(c):
    if c['quant'] not in META: raise ValueError('不支持的量化')
    for key,lo,hi in [('context',8192,262144),('reserve',700,65536),('workers',0,256),
                       ('parallel',1,8),('spec',2,8),('vision_tokens',1,16384),('port',1024,65535)]:
        if type(c[key]) is not int or not lo <= c[key] <= hi: raise ValueError(f'{key} 必须为 {lo}–{hi} 整数')
    for key,values in {'kv':['int8','q4_0','k8v4'],'streaming':['auto','on','off'],'cuda':['12','13'],
        'low_ram':['auto','on','off','resident','mmap'],'draft_vocab':['cjk','en','cyrillic','fr'],
        'vision':['no','gpu','cpu'],'host':['127.0.0.1','0.0.0.0'],
        'prefill':['auto','256','512','1024','2048','4096','8192','auto:16384','auto:32768']}.items():
        if c[key] not in values: raise ValueError(f'不支持的 {key}')
    if c['kv']=='k8v4' and c['streaming']=='on': raise ValueError('k8v4 不支持 KV streaming')
    if c['streaming']=='on' and c['context']<65536: raise ValueError('KV streaming 需要至少64K')
    if str(c['cache']) != 'auto' and not re.fullmatch(r'[1-9]\d{0,5}',str(c['cache'])): raise ValueError('expert_cache 须为 auto 或正整数条目数')
    if not 0 <= c['resident'] <= 1024: raise ValueError('RAM 预算无效')
    if c['resident'] and not c['quant'].startswith('UD-'): raise ValueError('手动 resident 预算仅适用于 Unsloth')
    if c['quant']=='UD-Q4_K_XL' and (not c['q4_ok'] or c['vision']!='no'):
        raise ValueError('UD-Q4_K_XL 需接受实验性且不支持这里的 vision 配置')
    if c['cuda']=='12' and not c['cuda12_ok']: raise ValueError('需明确接受 CUDA12 实验路径')

def hardware():
    mem={}
    if Path('/proc/meminfo').exists():
        for line in Path('/proc/meminfo').read_text().splitlines():
            k,v=line.split(':',1); mem[k]=int(v.split()[0])*1024
    gpu=[]
    try:
        p=subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total,memory.free,driver_version,compute_cap',
            '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=15,check=True)
        for line in p.stdout.splitlines():
            i,name,total,free,driver,cc=[s.strip() for s in line.split(',')]
            gpu.append(dict(index=int(i),name=name,total_mib=float(total),free_mib=float(free),driver=driver,cc=float(cc)))
    except (OSError,ValueError,subprocess.SubprocessError): pass
    cpu=Path('/proc/cpuinfo').read_text() if Path('/proc/cpuinfo').exists() else ''
    return dict(gpus=gpu,ram=mem.get('MemTotal',0),available=mem.get('MemAvailable',0),
                avx2=bool(re.search(r'\bavx2\b',cpu)),arch=platform.machine(),cpu_count=os.cpu_count())

def existing_bytes(path):
    if not path.exists(): return 0
    total=0
    for root,dirs,files in os.walk(path):
        for name in files:
            try: total += (Path(root)/name).stat().st_size
            except OSError: pass
    return total

def preflight(c,h,models):
    validate(c)
    blockers=[]; warnings=[]
    if h['arch'] not in ('x86_64','AMD64') or not h['avx2']: blockers.append('此 Notebook 要求 x86-64 AVX2 CPU')
    if not h['gpus']: blockers.append('未检测到 NVIDIA GPU（或 nvidia-smi 查询失败）')
    else:
        g=h['gpus'][0]
        minimum=(580,0,0) if c['cuda']=='13' else (525,60,13)
        d=tuple(map(int,g['driver'].split('.'))); d=(d+(0,0))[:3]
        if d<minimum: blockers.append(f"GPU 0 驱动 {g['driver']} 不满足 CUDA {c['cuda']} 最低检查 {minimum}")
        if g['cc'] < (7.5 if c['cuda']=='13' else 6.0): blockers.append('GPU 架构不满足所选 CUDA 路径')
        if g['total_mib']<8*1024: blockers.append('本 Notebook 的保守下限是 8 GiB 显存')
        if g['free_mib']<c['reserve']+4096: blockers.append('空闲显存不足：reserve 外至少需要4GiB；这只是下限，不保证能启动')
    download,ram_gb,arena=META[c['quant']]
    kvbytes=c['context']*13*{'int8':1056,'q4_0':576,'k8v4':816}[c['kv']]
    stream=c['context']>=65536 and c['kv']!='k8v4' and (c['streaming']=='on' or
        (c['streaming']=='auto' and h['ram']>=ram_gb*1e9+kvbytes+1e9))
    recommended=ram_gb*1e9+(kvbytes if stream else 0)+4e9
    if h['ram']<32e9: blockers.append('系统 RAM 少于32GB：本 Notebook 不继续这个完整基础模型部署')
    if h['available']<recommended:
        (warnings if c['low_ram_ok'] else blockers).append(f'可用 RAM 低于保守估计 {recommended/1e9:.1f}GB；需更多 RAM 或显式接受低RAM/OOM风险')
    # 额外15GB覆盖MTP/环境/元数据，vision再5GB；非UD打包通常另存arena
    pack=(max(arena,40) if not c['quant'].startswith('UD-') else 8)
    extra=15+(5 if c['vision']!='no' else 0)
    models.mkdir(parents=True,exist_ok=True)
    # 仅扣除此量化已下载的文件；不能用别的模型文件虚增已缓存量
    cached=sum(p.stat().st_size for p in models.rglob('*') if p.is_file() and c['quant'] in str(p))
    download_left=max(0,download*1e9-cached)
    same=os.stat(models).st_dev==os.stat(DATA).st_dev
    tag=('unsloth-' if c['quant'].startswith('UD-') else '')+c['quant'].lower()
    pack_left=max(0,pack*1e9-existing_bytes(DATA/'packs'/tag))
    local_required=(download_left if same else 0)+pack_left+extra*1e9
    free=shutil.disk_usage(DATA).free
    if free<local_required: blockers.append(f'本地磁盘需保守预留 {local_required/1e9:.1f}GB，现有 {free/1e9:.1f}GB；包含保守打包/构建余量')
    if not same:
        warnings.append(f'Drive 还需约 {download_left/1e9:.1f}GB；FUSE可用空间不等于账户配额，请人工确认')
    if c['cuda']=='12':
        nvcc=shutil.which('nvcc')
        ver=subprocess.run([nvcc,'--version'],capture_output=True,text=True).stdout if nvcc else ''
        version=re.search(r'release (12)\.(\d+)',ver)
        if version and h['gpus'] and h['gpus'][0]['cc']>=12 and int(version.group(2))<8: blockers.append('sm120需要CUDA12.8+工具链')
        if not version: blockers.append('CUDA12实验构建要求已安装的 nvcc12.x；当前未找到，改用兼容CUDA13的运行时或自行准备12.x工具链')
    if c['cuda']=='12': warnings.append('CUDA12 实验路径；最低驱动仅兼容下限，PTX/构建可能仍要求更新驱动；失败不自动改宿主驱动')
    return dict(blockers=blockers,warnings=warnings,download_GB=download,local_free_GB=round(free/1e9,1),
        estimated_additional_local_GB=round(local_required/1e9,1),KV_full_GiB=round(kvbytes/2**30,2),
        streaming_estimate=stream,context=c['context'],note='KV数字不含全部模型、临时prefill及并发内存；预设不能保证完整256K运行')

def redact(text):
    for value in [API_KEY,os.environ.get('HF_TOKEN',''),os.environ.get('HUGGING_FACE_HUB_TOKEN','')]:
        if value: text=text.replace(value,'[REDACTED]')
    text=re.sub(r'(?i)(Bearer\s+)[^\s"\']+',r'\1[REDACTED]',text)
    text=re.sub(r'hf_[A-Za-z0-9]{10,}','[REDACTED]',text)
    return text

def run_logged(argv, logfile, cwd=None, env=None):
    if not isinstance(argv,list) or not all(isinstance(x,str) for x in argv): raise TypeError('argv 必须为字符串列表')
    with open(logfile,'a',encoding='utf-8') as log:
        p=subprocess.Popen(argv,cwd=cwd,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        print('PID',p.pid,'日志:',Path(logfile).name)
        try:
            last_update=time.monotonic()
            while p.poll() is None:
                time.sleep(2)
                if time.monotonic()-last_update>=30:
                    print(redact('\n'.join(Path(logfile).read_text(errors='replace').splitlines()[-4:])))
                    last_update=time.monotonic()
        except BaseException:
            os.killpg(p.pid,signal.SIGTERM)
            try: p.wait(timeout=15)
            except subprocess.TimeoutExpired: os.killpg(p.pid,signal.SIGKILL); p.wait()
            raise
        if p.returncode:
            tail=Path(logfile).read_text(errors='replace').splitlines()[-35:]
            print(redact('\n'.join(tail)))
            raise RuntimeError(f'命令失败，退出码 {p.returncode}；见 {Path(logfile).name}')

def checkout(update=False):
    if not REPO.exists():
        run_logged(['git','clone','--no-checkout',ORIGIN,str(REPO)],BASE/'checkout.log')
    origin=subprocess.run(['git','remote','get-url','origin'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
    if origin!=ORIGIN: raise RuntimeError('已有目录的远端不匹配；不覆盖')
    ref='refs/heads/main' if update else PIN
    run_logged(['git','fetch','origin',ref],BASE/'checkout.log',cwd=REPO)
    run_logged(['git','checkout','--detach','FETCH_HEAD'],BASE/'checkout.log',cwd=REPO)
    actual=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
    if not update and actual!=PIN: raise RuntimeError('源码固定版本验证失败')
    print('源码 commit:',actual,'（更新模式未经过本 Notebook 固定版本测试）' if update else '')

def setup_command(c,python,models):
    validate(c)
    cmake=(REPO/'CMakeLists.txt').read_text()
    match=re.search(r'project\(strata VERSION ([\d.]+)',cmake)
    if not match: raise RuntimeError('无法确定固定release版本')
    flags={'--family':'unsloth' if c['quant'].startswith('UD-') else 'qwen','--model':c['quant'],
        '--context':c['context'],'--kv':c['kv'],'--kv-streaming':c['streaming'],'--cuda':c['cuda'],
        '--low-ram':c['low_ram'],'--vision':c['vision'],'--vision-tokens':c['vision_tokens'],
        '--vram-reserve-mib':c['reserve'],'--parallel':c['parallel'],'--draft-vocab':c['draft_vocab'],
        '--host':'127.0.0.1','--port':c['port'],'--gpu':'0','--backend':'cuda',
        '--data-dir':DATA,'--models-dir':models,'--experimental-speed-projection':'off',
        '--prebuilt':f'https://github.com/Niko1221/Strata/releases/download/v{match.group(1)}/'}
    if c['resident']: flags['--resident-budget-gib']=c['resident']
    return [python,str(REPO/'setup.py'),'--yes','--setup','--no-start','--no-browser']+[x for k,v in flags.items() for x in (k,str(v))]

def require_supported_setup(argv):
    source=(REPO/'setup.py').read_text()
    supported=set(re.findall(r'add_argument\("(--[\w-]+)"',source))
    unsupported=[a for a in argv if a.startswith('--') and a not in supported]
    if unsupported: raise RuntimeError('setup 不支持这些参数: '+str(unsupported))

def set_arg(args,key,value):
    out=[]; i=0
    while i<len(args):
        if args[i]==key: i+=2
        else: out.append(args[i]); i+=1
    return out+[key,str(value)]

INSTALL_KEYS = ('quant','context','kv','streaming','cuda','low_ram','resident','vision','vision_tokens','draft_vocab')
def install_selection(c):
    selection={k:c[k] for k in INSTALL_KEYS}
    selection['models_path']=str(MODELS.resolve())
    selection['source_head']=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True).stdout.strip()
    return selection
def save_install_selection(c):
    (BASE/'installed-selection.json').write_text(json.dumps(install_selection(c),sort_keys=True))

def prepared_config(c):
    try: installed=json.loads((BASE/'installed-selection.json').read_text())
    except (OSError,ValueError): raise RuntimeError('未验证此 Notebook 的安装完成；请运行安装单元')
    if installed!=install_selection(c): raise RuntimeError('安装相关选项已改变；重新运行安装单元以应用，现有下载会复用')
    tag=('unsloth-' if c['quant'].startswith('UD-') else '')+c['quant'].lower()
    src=REPO/f'strata-{tag}.json'
    if not src.exists(): raise RuntimeError('尚无安装配置；先完成安装')
    cfg=json.loads(src.read_text())
    args=list(cfg['args'])
    # 确认用户已对当前模型/上下文/KV/vision重新运行安装，不静默忽略旧设置
    def av(key,default=None): return args[args.index(key)+1] if key in args else default
    if int(av('--max-context','0'))!=c['context'] or (c['context']>8192 and av('--kv')!=c['kv']):
        raise RuntimeError('安装配置上下文/KV与表单不同；重新运行安装单元')
    if bool(cfg.get('vision'))!=(c['vision']!='no'): raise RuntimeError('vision已改变，需重新安装')
    for key,val in {'--expert-cache':c['cache'],'--vram-reserve-mib':c['reserve'],
                    '--prefill':c['prefill'],'--spec':c['spec'],'--kv':c['kv']}.items(): args=set_arg(args,key,val)
    if c['workers']: args=set_arg(args,'--pool-workers',c['workers'])
    cfg.update(args=args,host=c['host'],port=c['port'],parallel=c['parallel'],open_browser=False,
               log=str(BASE/'engine.log'),api_monitor=False)
    cfg.pop('api_key',None)
    cfg.pop('mcp_servers',None)
    cfg.pop('before_load',None)
    cfg.pop('scripts',None)
    return cfg

def proc_identity(pid):
    try:
        raw=Path(f'/proc/{pid}/stat').read_text(); rest=raw[raw.rfind(')')+2:].split()
        if rest[0]=='Z': return None
        return {'start':rest[19],'cmd':Path(f'/proc/{pid}/cmdline').read_bytes().decode().split('\0')[:-1]}
    except (OSError,ValueError): return None

def read_state():
    try: return json.loads(STATE.read_text())
    except (OSError,ValueError): return None

def owned_alive():
    s=read_state()
    try: return bool(s and proc_identity(s['pid'])==s['identity'] and os.getpgid(s['pid'])==s['pid'])
    except ProcessLookupError: return False

def stop_owned():
    s=read_state()
    if not s: print('没有记录的服务'); return
    if not owned_alive():
        previous=PROCESSES.pop(s['pid'],None)
        if previous and previous.poll() is not None: previous.wait()
        STATE.unlink(missing_ok=True); print('记录已过期；未终止任何其它进程'); return
    os.killpg(s['pid'],signal.SIGTERM)
    deadline=time.monotonic()+20
    while owned_alive() and time.monotonic()<deadline: time.sleep(.2)
    if owned_alive(): os.killpg(s['pid'],signal.SIGKILL)
    process=PROCESSES.pop(s['pid'],None)
    if process:
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: pass
    STATE.unlink(missing_ok=True)
    print('已停止本 Notebook 的服务；Colab运行时仍在占用资源')

def api(path,payload=None,timeout=30,port=None):
    if port is None:
        state=read_state()
        if not state or not owned_alive(): raise RuntimeError('本 Notebook 未启动服务')
        port=state['port']
    body=None if payload is None else json.dumps(payload).encode()
    r=Request(f'http://127.0.0.1:{port}{path}',data=body,headers={'Authorization':'Bearer '+API_KEY,'Content-Type':'application/json'})
    with urlopen(r,timeout=timeout) as resp: return json.load(resp)

def launch_owned(argv,port,fingerprint,env=None):
    if owned_alive():
        s=read_state()
        if s['fingerprint']!=fingerprint or s['port']!=port: raise RuntimeError('服务配置改变；请选择重启')
        return s
    with socket.socket() as sock:
        sock.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
        try: sock.bind(('127.0.0.1',port))
        except OSError: raise RuntimeError('端口已被占用，不会终止未知进程；换端口或自行确认')
    with open(BASE/'server.log','a') as log:
        p=subprocess.Popen(argv,cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    PROCESSES[p.pid]=p
    ident=proc_identity(p.pid)
    if not ident: raise RuntimeError('服务在记录归属前退出；查看日志')
    s={'pid':p.pid,'identity':ident,'port':port,'fingerprint':fingerprint}
    STATE.write_text(json.dumps(s)); STATE.chmod(0o600)
    return s

def await_ready(port,timeout):
    if not 1<=timeout<=7200: raise ValueError('等待时间须为1–7200秒')
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        if not owned_alive(): raise RuntimeError('服务提前退出；运行诊断单元')
        try:
            h=api('/health',timeout=3,port=port)
            if h.get('status')=='ok' and h.get('loaded') is True: return h
        except (OSError,ValueError): pass
        time.sleep(2)
    raise TimeoutError('尚未就绪，服务未被终止。可查看日志、继续等待，或用停止单元释放模型资源')

def start_service(c,timeout=900):
    cfg=prepared_config(c)
    fingerprint=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()+API_KEY.encode()).hexdigest()
    runtime=BASE/'colab-runtime.json'
    if not owned_alive(): runtime.write_text(json.dumps(cfg,indent=2)); runtime.chmod(0o600)
    env=os.environ.copy(); env['STRATA_API_KEY']=API_KEY
    env.pop('STRATA_DEBUG_REQUESTS',None)
    cmd=[str(REPO/'.venv/bin/python'),str(REPO/'serve/server.py'),'--engine','strata','--config',str(runtime),
         '--host',c['host'],'--port',str(c['port'])]
    launch_owned(cmd,c['port'],fingerprint,env)
    h=await_ready(c['port'],timeout)
    if h.get('max_context')!=c['context']: raise RuntimeError('服务的上下文与配置不一致，请检查 health/日志')
    print('已就绪（非完整长测）:',h.get('model'),'context=',h.get('max_context'),'loaded=',h.get('loaded'))
    print(f"同一Colab运行时API: http://127.0.0.1:{c['port']}/v1；密钥变量名 API_KEY，不打印")


In [ ]:
#@title 3 · 可选 Drive 缓存（默认不挂载）
mount_drive = False #@param {type:"boolean"}
use_drive_model_cache = False #@param {type:"boolean"}
if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
if use_drive_model_cache:
    if not mount_drive or not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError("先明确勾选挂载 Drive 并完成授权")
    MODELS = Path('/content/drive/MyDrive/Strata-model-cache')
else:
    MODELS = DATA / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
print("模型缓存:", MODELS)
print("Drive 保存 GGUF；打包目录仍在本地 SSD，但运行可能随机读取 Drive 的 GGUF，尤其 PLE/Unsloth。追求速度请用本地缓存；Drive 配额需自行确认。不要保存令牌或日志到共享目录。")


In [ ]:
#@title 4 · 硬件、预算和参数检查（默认可安全运行）
CFG = dict(quant=model_quant, context=context_tokens, kv=kv_precision, streaming=kv_streaming,
    cuda=cuda_version, cuda12_ok=accept_experimental_cuda12, low_ram=low_ram,
    resident=resident_budget_gib, cache=expert_cache, reserve=vram_reserve_mib,
    workers=cpu_pool_workers, prefill=prefill_chunk, parallel=parallel_requests,
    spec=spec_window, draft_vocab=draft_vocab, vision=vision, vision_tokens=vision_tokens,
    host=api_host, port=api_port, low_ram_ok=accept_low_ram_risk, q4_ok=accept_experimental_q4)
validate(CFG)
HW = hardware()
REPORT = preflight(CFG, HW, MODELS)
print(json.dumps(REPORT, ensure_ascii=False, indent=2))
print("检测 GPU:", HW['gpus'], "RAM GiB:", round(HW['ram']/2**30, 1), "可用 RAM GiB:", round(HW['available']/2**30, 1))
print("下一步前阻塞项必须清空。256K 预算是估计，不是完整运行保证；实际以 setup 和 /health 为准。")


In [ ]:
#@title 5 · 获取固定源码（小下载，不下载模型）
fetch_source = False #@param {type:"boolean"}
update_to_latest = False #@param {type:"boolean"}
if fetch_source:
    if owned_alive():
        raise RuntimeError("先停止本 Notebook 的服务，再更新源码")
    checkout(update_to_latest)
else:
    print("已跳过源码获取；需要时勾选 fetch_source")


默认固定代码 commit，并把预编译包固定到该代码的 release 目录；不主动追踪 latest。选择 update_to_latest 会切到当前 main，兼容性未经过本 Notebook 的固定版测试；需重新检查/安装。上游下载器对失效 Hugging Face revision 存在回退 main 的行为，因此并非所有权重的完全离线锁定；保留日志核查。


In [ ]:
#@title 6 · 安装 + 可续传下载（重大耗时动作，独立确认）
install_and_download = False #@param {type:"boolean"}
accept_download_storage_and_build = False #@param {type:"boolean"}
if install_and_download:
    if not accept_download_storage_and_build:
        raise RuntimeError("确认数十至上百 GB 下载/磁盘占用，及可能的依赖安装/编译后再勾选")
    if owned_alive():
        raise RuntimeError("服务运行中，先停止再修改安装")
    validate(CFG)
    REPORT = preflight(CFG, hardware(), MODELS)
    if REPORT['blockers']:
        raise RuntimeError("硬件/预算检查未通过: " + '; '.join(REPORT['blockers']))
    if not (REPO/'setup.py').is_file():
        raise RuntimeError("先获取源码")
    # 独立 venv；不修改 Colab 自带 torch/CUDA Python 环境
    py = REPO/'.venv/bin/python'
    venv_ready = py.exists() and subprocess.run([str(py), '-m', 'pip', '--version'],
        capture_output=True, timeout=30).returncode == 0
    if not venv_ready:
        uv = shutil.which('uv')
        if uv:
            run_logged([uv, 'venv', '--allow-existing', '--seed', '--python', sys.executable,
                str(REPO/'.venv')], BASE/'venv.log')
        else:
            run_logged([sys.executable, '-m', 'venv', str(REPO/'.venv')], BASE/'venv.log')
    argv = setup_command(CFG, str(py), MODELS)
    require_supported_setup(argv)
    print("开始，输出保存在本地 install.log；可在诊断单元看脱敏尾部。中断会终止本次安装进程组，下次重跑同配置续传。")
    run_logged(argv, BASE/'install.log', cwd=REPO)
    save_install_selection(CFG)
    print("安装结束。再运行下一单元应用配置并启动。")
else:
    print("已跳过安装/下载")


In [ ]:
#@title 7 · 应用配置并启动 / 停止 / 重启
server_action = "不操作" #@param ["不操作", "启动", "停止", "重启"]
startup_timeout_seconds = 900 #@param {type:"integer"}
if server_action == '停止':
    if 'stop_external' in globals(): stop_external(False)
    stop_owned()
elif server_action in ('启动', '重启'):
    validate(CFG)
    if server_action == '重启':
        if 'stop_external' in globals(): stop_external(False)
        stop_owned()
    start_service(CFG, timeout=startup_timeout_seconds)
else:
    print("已跳过服务操作")


服务启动使用单独的 colab-runtime.json，不直接覆盖上游生成配置。相同配置重复启动会复用本 Notebook 的进程；配置变了需选择重启。停止只处理已记录且 Linux 进程出生时间、参数均匹配的进程组，端口被其它程序占用不会杀它。

密钥自动生成，仅留在当前 Python 内存，通过环境传入，不输出。无论 host 如何都启用 API key。0.0.0.0 只是绑定网卡，**不创建外网访问**；外部访问请用第11–13节的独立认证网关，不能直接暴露原服务。Colab重启后需重新执行这些单元。


In [ ]:
#@title 8 · 短聊天 / 采样 / 推理 / 工具声明（勾选才推理）
run_chat = False #@param {type:"boolean"}
prompt = "请用三句话介绍你自己。" #@param {type:"string"}
system_prompt = "你是一个有帮助的助手。" #@param {type:"string"}
max_output_tokens = 256 #@param {type:"integer"}
temperature = 0.6 #@param {type:"number"}
top_p = 0.95 #@param {type:"number"}
top_k = 20 #@param {type:"integer"}
min_p = 0.0 #@param {type:"number"}
repetition_penalty = 1.0 #@param {type:"number"}
presence_penalty = 0.0 #@param {type:"number"}
frequency_penalty = 0.0 #@param {type:"number"}
penalty_last_n = 64 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
reasoning_effort = "none" #@param ["none", "low", "medium", "high"]
reasoning_budget_tokens = 0 #@param {type:"integer"}
request_timeout_seconds = 600 #@param {type:"integer"}
include_demo_tool = False #@param {type:"boolean"}
if run_chat:
    if not 1 <= max_output_tokens < CFG['context']:
        raise ValueError("输出上限必须大于零且小于总上下文")
    if not (0 <= temperature <= 2 and 0 < top_p <= 1 and 0 <= min_p <= 1 and 1 <= top_k <= 64 and reasoning_budget_tokens >= 0 and repetition_penalty > 0 and presence_penalty >= 0 and frequency_penalty >= 0 and penalty_last_n >= 0 and seed > 0):
        raise ValueError("采样/推理参数超出本 Notebook 的安全范围")
    req = dict(model='strata', messages=[{'role':'system','content':system_prompt}, {'role':'user','content':prompt}],
        max_tokens=max_output_tokens, temperature=temperature, top_p=top_p, top_k=top_k, min_p=min_p,
        reasoning_effort=reasoning_effort, reasoning_budget_tokens=reasoning_budget_tokens,
        repetition_penalty=repetition_penalty, presence_penalty=presence_penalty, frequency_penalty=frequency_penalty,
        penalty_last_n=penalty_last_n, seed=seed, stream=False)
    if include_demo_tool:
        req['tools']=[{'type':'function','function':{'name':'get_weather', 'description':'演示工具声明，不执行',
            'parameters':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}}]
        req['tool_choice']='auto'
    t=time.monotonic()
    result=api('/v1/chat/completions', req, timeout=request_timeout_seconds)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("总耗时秒:", round(time.monotonic()-t,2))
else:
    print("未推理。工具调用只返回模型的请求，本 Notebook 不执行工具。")


后续聊天可修改 prompt 再运行；此单元每次发送一个独立对话，若要连续上下文，把此前的 messages 和回复显式保留并再次发送。模型输出可能含 reasoning_content，按返回 JSON 区分思考/最终答复。输出 token 预算包含推理消耗。

同一个 Colab 运行时中的客户端：base_url = http://127.0.0.1:8080/v1（若改端口请替换），api_key = API_KEY，model = strata。勿把 API_KEY 打印、截图或粘到公开 Notebook。Vision 需安装时启用，可按上游 OpenAI image_url 内容格式提交；本 Notebook 不自动上传你的图片。


In [ ]:
#@title 9 · 长上下文压力测试（可选，可能很慢且消耗大量 Compute Units）
run_long_context_test = False #@param {type:"boolean"}
accept_long_test_cost = False #@param {type:"boolean"}
long_test_target_tokens = 32768 #@param {type:"integer"}
long_test_timeout_seconds = 3600 #@param {type:"integer"}
if run_long_context_test:
    if not accept_long_test_cost:
        raise RuntimeError("先确认时间/CU 成本；建议 32K→64K→128K→接近256K，逐档测试")
    if not 1024 <= long_test_target_tokens <= CFG['context']-1024:
        raise ValueError("目标需保留至少 1024 token 给模板差异与输出；256K 总窗口不能再塞满256K输入")
    marker='NEEDLE-731928'
    prefix='请在文末回答开头的标识符，只输出标识符。标识符：'+marker+'。\n'
    suffix='\n开头标识符是什么？'
    def sample(n): return prefix + ('这是一段与答案无关的背景材料。\n'*n) + suffix
    def count(n):
        return api('/v1/messages/count_tokens', {'model':'strata','max_tokens':32,
            'messages':[{'role':'user','content':sample(n)}], 'thinking':{'type':'disabled'}}, timeout=120)['input_tokens']
    lo, hi = 0, long_test_target_tokens
    while lo < hi:
        mid=(lo+hi+1)//2
        if count(mid) <= long_test_target_tokens: lo=mid
        else: hi=mid-1
    measured=count(lo)
    print("服务端 Anthropic 模板计数:", measured, "token；仅测容量/一次检索，不代表长文理解质量。")
    start=time.monotonic()
    out=api('/v1/messages', {'model':'strata','max_tokens':32,'temperature':0,
        'messages':[{'role':'user','content':sample(lo)}], 'thinking':{'type':'disabled'}}, timeout=long_test_timeout_seconds)
    print(json.dumps(out,ensure_ascii=False,indent=2))
    print('总耗时秒:',round(time.monotonic()-start,2),'预期标识符:',marker)
else:
    print("未运行长测试。HTTP 超时可能发生在计算仍进行时；需取消则用服务停止单元。")


In [ ]:
#@title 10 · 脱敏诊断（不打印环境变量、令牌、完整配置）
show_diagnostics = False #@param {type:"boolean"}
log_tail_lines = 40 #@param {type:"integer"}
if show_diagnostics:
    print('Python:',sys.version.split()[0], 'GPU/RAM:', hardware())
    print('本 Notebook 服务归属:', owned_alive())
    if REPO.exists():
        print('代码 commit:', subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, capture_output=True,text=True).stdout.strip())
    try: print('health:', api('/health', timeout=5))
    except Exception as e: print('health 不可用:',type(e).__name__)
    for p in [BASE/'install.log', BASE/'server.log', BASE/'engine.log']:
        if p.exists():
            print('---',p.name,'（日志可能含提示词，请分享前人工检查）---')
            print(redact('\n'.join(p.read_text(errors='replace').splitlines()[-min(max(log_tail_lines,1),200):])))
else:
    print("按需启用诊断")


## 故障速查与边界
- 没有 GPU / driver < 580：重新选 GPU 运行时；或显式选择实验 CUDA 12，确认兼容风险。Notebook 不替换 Colab 宿主驱动
- RAM 不够：先换高 RAM；必要时 Q2_0、降低 context、关闭 vision/并发，或显式尝试 low_ram。不能把显存当作等量系统 RAM
- 显存 OOM：减小上下文、prefill 分块/并发；q4_0 KV 或足够 RAM 下 streaming；expert_cache 保持 auto。减少 reserve 会挤占其它组件内存，不能盲目降到零
- 磁盘不足：检查本地打包空间和 Drive 配额；不要删除正在使用的 pack。Notebook 不自动删除模型
- 下载中断：重跑相同安装单元；上游续传并校验。关闭运行时会丢失 /content，Drive 只保留勾选缓存的 GGUF
- 启动超时：进程保留以便诊断，查看日志；相同配置再启动继续等候，或停止后调整。安装/启动成功与完整 256K 推理成功是不同验证阶段
- 重启 runtime 后：/content 可能仍在或可能被清空，重新从第 1 单元运行；API 密钥重新生成，不依赖以前输出
- Colab 资源、会话时长和 CU 消耗以账户界面显示为准。测试无保活，不规避闲置/使用限制

此 Notebook 仅管理自己启动的进程，不操作你的本机；没有自动执行模型提出的工具调用；外部入口仅在第12节明确开启时创建。不要在 Notebook 输出中放 HF token、Drive 私密链接或 API key。


## 11–13 · 可选：让本地客户端连接 Colab 外部 API

默认关闭。这里建立**临时 ngrok HTTPS 入口 → 仅API安全网关 → Colab内Strata**；不会公开 Strata 网页、配置管理或 MCP。只开放 health、models 和 chat/completions，三者都要求 Bearer key。

**先确认隐私与费用**
- HTTPS 在 ngrok 边缘终止，ngrok 是第三方流量处理方，会处理你的提示词/响应；这不是对 ngrok 端到端保密。不要未经授权传输敏感数据。关闭请求检查不是“供应商看不到数据”
- 仅用于你正在操作 Notebook 的临时计算访问，不是24小时托管。Colab 付费且余额为正时相关远程使用限制不同；仍须遵守[Colab FAQ](https://research.google.com/colaboratory/faq.html)，不提供保活/限制规避
- 除 Colab GPU/CU，ngrok 也有账户配额/费用。免费套餐可能限制流量和请求数；大上下文特别容易消耗流量。以[ngrok 当前限制](https://ngrok.com/docs/pricing-limits/free-plan-limits)与账户界面为准
- pyngrok 固定为 8.1.2，首次启动会从 ngrok 官方下载当前 v3 二进制；二进制不是完整版本锁定

**准备两个已有凭证**（只在 Colab 左侧钥匙图标 Secrets 添加并开启此Notebook访问）：
1. NGROK_AUTHTOKEN：你自己的 ngrok 账户令牌
2. STRATA_API_KEY：你自行生成的至少32字符随机值，仅字母、数字、下划线和短横线。把同一值安全设置到本地客户端环境变量 STRATA_API_KEY；不要写进 Notebook 代码、输出、聊天或会话 JSON

ngrok 令牌只写到当前运行时私有0600临时配置，不写Drive、不进命令行，停止时删除；内核崩溃后文件可能保留到运行时删除，重跑停止可清理。本 Notebook 不创建账号/令牌，不保存新的长期授权。


In [ ]:
#@title 11 · 外部API工具（只定义函数，默认不联网）
# 外部 API 安全网关：标准库实现。此单元只定义函数，默认不安装、不联网、不启动。
import threading, queue, hmac, http.client, urllib.parse, datetime, logging
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
EXTERNAL = globals().get('EXTERNAL', {})
EXTERNAL_LOCK = globals().get('EXTERNAL_LOCK', threading.RLock())
EXT_DIR = BASE/'external-api'
EXT_STATE = BASE/'owned-tunnel.json'
SESSION_FILE = BASE/'strata-session.json'
NGROK_VERSION = '8.1.2'


def valid_public_url(value):
    u=urllib.parse.urlsplit(value)
    host=(u.hostname or '').lower()
    allowed=('ngrok-free.app','ngrok.app','ngrok.io')
    if (u.scheme!='https' or not host or u.username or u.password or u.port not in (None,443)
        or u.path not in ('','/') or u.query or u.fragment or not any(host.endswith('.'+x) for x in allowed)):
        raise ValueError('隧道未返回可信的 ngrok HTTPS 主机；不会导出或发送密钥')
    return 'https://'+host


def session_manifest(public_url,context,minutes,model):
    now=datetime.datetime.now(datetime.timezone.utc)
    return dict(schema_version=1,base_url=valid_public_url(public_url)+'/v1',model=model,max_context=context,
        api_key_env='STRATA_API_KEY',created_at=now.isoformat(),expires_at=(now+datetime.timedelta(minutes=minutes)).isoformat(),
        streaming=True,endpoints={'health':'/health','models':'/v1/models','chat':'/v1/chat/completions'},
        note='temporary Colab session; old URL invalid after restart')


class StrataGateway(ThreadingHTTPServer):
    daemon_threads=True
    allow_reuse_address=True
    def __init__(self,address,backend_port,backend_key,public_key,body_limit=8*1024*1024,
                 concurrency=1,stream_timeout=3600,nonstream_timeout=240,heartbeat=15,backend_guard=None):
        if not (1024<=backend_port<=65535 and 1<=concurrency<=8 and 1<=body_limit<=16*1024*1024
                and 1<=nonstream_timeout<=240 and 1<=stream_timeout<=7200 and 0<heartbeat<=30):
            raise ValueError('网关参数越界')
        self.backend_port=backend_port; self.backend_key=backend_key; self.public_key=public_key
        self.backend_guard=backend_guard or (lambda:True)
        self.body_limit=body_limit; self.slots=threading.BoundedSemaphore(concurrency)
        self.stream_timeout=stream_timeout; self.nonstream_timeout=nonstream_timeout; self.heartbeat=heartbeat
        self.active_connections=set(); self.active_sockets=set(); self.connection_lock=threading.RLock()
        self.closing=threading.Event()
        self.connection_slots=threading.BoundedSemaphore(max(8,concurrency*4))
        super().__init__(address,GatewayHandler)
    def process_request(self,request,client_address):
        if not self.connection_slots.acquire(blocking=False):
            self.shutdown_request(request);return
        try:super().process_request(request,client_address)
        except BaseException:self.connection_slots.release();raise
    def process_request_thread(self,request,client_address):
        try:super().process_request_thread(request,client_address)
        finally:self.connection_slots.release()
    def close_upstreams(self):
        with self.connection_lock:
            self.closing.set()
            for sock in list(self.active_sockets):
                try:sock.shutdown(socket.SHUT_RDWR)
                except OSError:pass
            for conn in list(self.active_connections):
                try:
                    if conn.sock:conn.sock.shutdown(socket.SHUT_RDWR)
                    conn.close()
                except OSError: pass
    def add_socket(self,sock):
        with self.connection_lock:self.active_sockets.add(sock)
    def remove_socket(self,sock):
        with self.connection_lock:self.active_sockets.discard(sock)
    def add_connection(self,conn):
        with self.connection_lock:self.active_connections.add(conn)
    def remove_connection(self,conn):
        with self.connection_lock:
            self.active_connections.discard(conn)
            self.active_sockets.discard(getattr(conn,'registered_socket',None))


class GatewayHandler(BaseHTTPRequestHandler):
    protocol_version='HTTP/1.1'
    server_version='StrataGateway'
    def __getattr__(self,name):
        if name.startswith('do_'):return lambda:self.handle_route(self.command)
        raise AttributeError(name)
    def log_message(self,*args): pass  # 不记录Authorization/提示词/URL查询
    def setup(self):
        super().setup(); self.connection.settimeout(30)
    def json_response(self,status,obj):
        body=json.dumps(obj,separators=(',',':')).encode()
        self.send_response(status); self.send_header('Content-Type','application/json')
        self.send_header('Content-Length',str(len(body))); self.send_header('Cache-Control','no-store')
        self.send_header('Connection','close'); self.end_headers()
        self.close_connection=True
        try:self.wfile.write(body)
        except (OSError,TimeoutError):pass
    def error(self,status,code):
        self.json_response(status,{'error':{'message':code,'type':'gateway_error','code':code}})
    def authorized(self):
        values=self.headers.get_all('Authorization',[])
        expected='Bearer '+self.server.public_key
        return len(values)==1 and hmac.compare_digest(values[0].encode(),expected.encode())
    def do_GET(self):self.handle_route('GET')
    def do_POST(self):self.handle_route('POST')
    def do_OPTIONS(self):self.handle_route('OPTIONS')
    def do_PUT(self):self.handle_route('PUT')
    def do_DELETE(self):self.handle_route('DELETE')
    def do_PATCH(self):self.handle_route('PATCH')
    def do_HEAD(self):self.handle_route('HEAD')
    def handle_route(self,method):
        if not self.authorized():self.error(401,'unauthorized');return
        if self.server.closing.is_set() or not self.server.backend_guard():self.error(503,'backend_session_ended');return
        allowed={('GET','/health'),('GET','/v1/models'),('POST','/v1/chat/completions')}
        if self.requestline.split()[1]!=self.path or (method,self.path) not in allowed:self.error(404,'route_not_available');return
        if self.headers.get('Transfer-Encoding') or len(self.headers.get_all('Content-Length',[]))>1:
            self.error(400,'invalid_framing');return
        try:n=int(self.headers.get('Content-Length','0'))
        except ValueError:self.error(400,'invalid_content_length');return
        if n<0 or n>self.server.body_limit:self.error(413,'request_too_large');return
        if method=='GET' and n:self.error(400,'unexpected_body');return
        if not self.server.slots.acquire(blocking=False):self.error(429,'gateway_busy');return
        try:
            body=b'';stream=False
            if method=='POST':
                if self.headers.get_content_type()!='application/json':self.error(415,'json_required');return
                try:
                    body=self.rfile.read(n)
                    if len(body)!=n:raise ValueError()
                    obj=json.loads(body)
                    if not isinstance(obj,dict) or not isinstance(obj.get('messages'),list):raise ValueError()
                    if not isinstance(obj.get('stream',False),bool):raise ValueError()
                    stream=obj.get('stream') is True
                except (OSError,ValueError):self.error(400,'invalid_json_request');return
            if self.server.closing.is_set() or not self.server.backend_guard():self.error(503,'backend_session_ended');return
            if stream:self.stream_proxy(body)
            else:self.json_proxy(method,body)
        except ConnectionAbortedError:self.error(503,'backend_session_ended')
        finally:self.server.slots.release()
    def connection_to_backend(self,timeout):
        server=self.server
        class GuardedConnection(http.client.HTTPConnection):
            def connect(connection):
                with server.connection_lock:
                    if server.closing.is_set() or not server.backend_guard():raise ConnectionAbortedError('backend ended')
                    connection.timeout=min(timeout,3)
                    super().connect()
                    connection.timeout=timeout;connection.sock.settimeout(timeout)
                    connection.registered_socket=connection.sock;server.active_sockets.add(connection.sock)
        if server.closing.is_set() or not server.backend_guard():raise ConnectionAbortedError('backend ended')
        conn=GuardedConnection('127.0.0.1',server.backend_port,timeout=timeout)
        server.add_connection(conn)
        return conn
    def upstream_headers(self):
        return {'Authorization':'Bearer '+self.server.backend_key,'Content-Type':'application/json','Connection':'close'}
    def json_proxy(self,method,body):
        conn=self.connection_to_backend(self.server.nonstream_timeout)
        hard_deadline=None;upstream_socket=None;timed_out=threading.Event();deadline=time.monotonic()+self.server.nonstream_timeout
        try:
            conn.request(method,self.path,body=body or None,headers=self.upstream_headers())
            upstream_socket=conn.sock
            self.server.add_socket(upstream_socket)
            def abort_slow_upstream():
                timed_out.set()
                if upstream_socket:
                    try:upstream_socket.shutdown(socket.SHUT_RDWR)
                    except OSError:pass
            hard_deadline=threading.Timer(max(.01,deadline-time.monotonic()),abort_slow_upstream)
            hard_deadline.daemon=True;hard_deadline.start()
            response=conn.getresponse()
            if response.status!=200:
                self.error(504 if response.status==504 else 502,'upstream_request_failed');return
            data=response.read(self.server.body_limit+1)
            if len(data)>self.server.body_limit:self.error(502,'upstream_response_too_large');return
            obj=json.loads(data)
            if self.path=='/health':
                obj={k:obj[k] for k in ('status','loaded','model','max_context') if k in obj}
            self.json_response(200,obj)
        except (socket.timeout,TimeoutError):self.error(504,'upstream_timeout')
        except (OSError,ValueError,http.client.HTTPException):
            self.error(504 if timed_out.is_set() else 502,'upstream_timeout' if timed_out.is_set() else 'upstream_unavailable')
        finally:
            if hard_deadline:hard_deadline.cancel()
            self.server.remove_socket(upstream_socket)
            conn.close();self.server.remove_connection(conn)
    def stream_proxy(self,body):
        # 先建立 SSE 再等待上游：长prefill时发注释心跳，避免ngrok空闲传输超时。
        # 若上游之后拒绝，HTTP状态已200，改用data:{error:...}再[DONE]传递错误。
        conn=self.connection_to_backend(self.server.stream_timeout)
        self.send_response(200);self.send_header('Content-Type','text/event-stream')
        self.send_header('Cache-Control','no-cache, no-store');self.send_header('X-Accel-Buffering','no')
        self.send_header('Connection','close');self.end_headers();self.close_connection=True
        pending=queue.Queue(maxsize=16); cancel=threading.Event(); upstream={}
        def publish(item):
            while not cancel.is_set():
                try:pending.put(item,timeout=.2);return
                except queue.Full:pass
        def worker():
            response=None
            try:
                conn.request('POST',self.path,body=body,headers=self.upstream_headers())
                upstream['socket']=conn.sock;self.server.add_socket(conn.sock)
                response=conn.getresponse()
                if response.status!=200 or 'text/event-stream' not in response.getheader('Content-Type',''):
                    publish(('error','upstream_stream_failed'));return
                while not cancel.is_set():
                    data=response.read1(65536)
                    if not data:break
                    publish(('data',data))
            except (socket.timeout,TimeoutError):publish(('error','upstream_timeout'))
            except (OSError,ValueError,http.client.HTTPException):publish(('error','upstream_unavailable'))
            finally:
                publish(('end',None))
                if response:response.close()
                self.server.remove_socket(upstream.get('socket'))
                conn.close();self.server.remove_connection(conn)
        worker_thread=threading.Thread(target=worker,daemon=True,name='strata-upstream')
        worker_thread.start()
        deadline=time.monotonic()+self.server.stream_timeout
        try:
            self.wfile.write(b': gateway connected\n\n');self.wfile.flush()
            while time.monotonic()<deadline:
                try:kind,value=pending.get(timeout=min(self.server.heartbeat,max(.01,deadline-time.monotonic())))
                except queue.Empty:
                    self.wfile.write(b': gateway heartbeat\n\n');self.wfile.flush();continue
                if kind=='end':break
                if kind=='error':
                    err=json.dumps({'error':{'message':value,'type':'gateway_error','code':value}}).encode()
                    self.wfile.write(b'data: '+err+b'\n\ndata: [DONE]\n\n');self.wfile.flush();break
                self.wfile.write(value);self.wfile.flush()
            else:
                self.wfile.write(b'data: {"error":{"message":"gateway_timeout","type":"gateway_error"}}\n\ndata: [DONE]\n\n');self.wfile.flush()
        except (OSError,TimeoutError):pass
        finally:
            cancel.set()
            sock=upstream.get('socket') or conn.sock
            if sock:
                try:sock.shutdown(socket.SHUT_RDWR)
                except OSError:pass
            conn.close();worker_thread.join(timeout=2)


def start_gateway(backend_port,public_key,body_mib=8,concurrency=1,stream_timeout=3600):
    snapshot=read_state() or {}
    def same_backend():
        current=read_state() or {}
        return bool(snapshot.get('identity') and snapshot.get('pid')==current.get('pid')
                    and proc_identity(snapshot['pid'])==snapshot['identity'] and owned_alive())
    gateway=StrataGateway(('127.0.0.1',0),backend_port,API_KEY,public_key,body_limit=body_mib*1024*1024,
                          concurrency=concurrency,stream_timeout=stream_timeout,backend_guard=same_backend)
    thread=threading.Thread(target=gateway.serve_forever,daemon=True,name='strata-gateway')
    thread.start()
    return gateway


def stop_recorded_tunnel():
    try:s=json.loads(EXT_STATE.read_text())
    except (OSError,ValueError):return
    pid=s.get('pid'); identity=s.get('identity')
    if isinstance(pid,int) and proc_identity(pid)==identity:
        os.kill(pid,signal.SIGTERM)
        end=time.monotonic()+5
        while proc_identity(pid)==identity and time.monotonic()<end:time.sleep(.1)
        if proc_identity(pid)==identity:os.kill(pid,signal.SIGKILL)
    EXT_STATE.unlink(missing_ok=True)


def _stop_external(stop_server=False):
    timer=EXTERNAL.pop('timer',None)
    if timer:timer.cancel()
    config=EXTERNAL.pop('ngrok_config',None)
    if config:
        try:
            from pyngrok import ngrok
            ngrok.kill(pyngrok_config=config)
        except Exception:pass  # 下方根据出生时间验证后回收；不打印可能含认证信息的异常
    stop_recorded_tunnel()
    gateway=EXTERNAL.pop('gateway',None)
    if gateway:
        gateway.close_upstreams();gateway.shutdown();gateway.server_close()
    EXTERNAL.clear()
    SESSION_FILE.unlink(missing_ok=True)
    (EXT_DIR/'ngrok.yml').unlink(missing_ok=True)
    if stop_server:stop_owned()
    print('外部API已停止，旧会话URL不再用于连接。'+('本Notebook服务也已停止。' if stop_server else 'Strata/Colab运行时仍可能占用GPU与CU。'))


def _start_external(minutes=60,body_mib=8,concurrency=1,stream_timeout=3600,stop_server_at_expiry=True):
    if minutes not in (15,30,60,120,240) or body_mib not in (1,2,4,8,16):raise ValueError('会话时长/体积选项无效')
    if type(concurrency) is not int or not 1<=concurrency<=8 or type(stream_timeout) is not int or not 1<=stream_timeout<=7200:
        raise ValueError('并发须1–8，流式超时须1–7200秒；尚未读取Secrets')
    if EXTERNAL.get('gateway'):
        raise RuntimeError('已有外部会话；先停止外部API再重新启动，避免留下多个入口')
    if not owned_alive():raise RuntimeError('先完成安装并启动Strata')
    health=api('/health',timeout=10)
    if health.get('loaded') is not True:raise RuntimeError('Strata尚未就绪')
    from google.colab import userdata
    try:
        public_key=userdata.get('STRATA_API_KEY')
        token=userdata.get('NGROK_AUTHTOKEN')
    except Exception:raise RuntimeError('请在Colab Secrets添加并授权 NGROK_AUTHTOKEN 与 STRATA_API_KEY；不要粘贴到代码/输出') from None
    if not isinstance(public_key,str) or not re.fullmatch(r'[A-Za-z0-9_-]{32,256}',public_key):
        raise ValueError('STRATA_API_KEY需为至少32字符随机值，仅字母数字-_；不能使用短密码')
    if not isinstance(token,str) or not token.strip():raise ValueError('NGROK_AUTHTOKEN为空')
    from pyngrok import ngrok
    from pyngrok.conf import PyngrokConfig
    logging.getLogger('pyngrok').setLevel(logging.CRITICAL)
    stop_recorded_tunnel()
    EXT_DIR.mkdir(parents=True,exist_ok=True);EXT_DIR.chmod(0o700)
    with socket.socket() as s:s.bind(('127.0.0.1',0));agent_port=s.getsockname()[1]
    config_path=EXT_DIR/'ngrok.yml'
    config_path.write_text(f'version: "2"\nweb_addr: 127.0.0.1:{agent_port}\nlog_level: warn\nauthtoken: '+json.dumps(token)+'\n')
    config_path.chmod(0o600)
    # pyngrok的auth_token字段会进入argv；改用临时0600配置，停止时删除，绝不保存到Drive。
    conf=PyngrokConfig(ngrok_path=str(EXT_DIR/'ngrok'),config_path=str(config_path),auth_token=None,
                      monitor_thread=False,startup_timeout=30)
    conf.auth_token=None  # 覆盖构造器可能读取的NGROK_AUTHTOKEN环境变量，防止传入argv
    EXTERNAL.update(ngrok_config=conf)
    try:
        gateway=start_gateway(read_state()['port'],public_key,body_mib,concurrency,min(stream_timeout,minutes*60))
        EXTERNAL['gateway']=gateway
        tunnel=ngrok.connect(addr=f'http://127.0.0.1:{gateway.server_port}',proto='http',schemes=['https'],inspect=False,pyngrok_config=conf)
        process=ngrok.get_ngrok_process(pyngrok_config=conf).proc
        identity=proc_identity(process.pid)
        if not identity:raise RuntimeError('隧道进程未保持运行')
        EXT_STATE.write_text(json.dumps({'pid':process.pid,'identity':identity}));EXT_STATE.chmod(0o600)
        manifest=session_manifest(tunnel.public_url,health['max_context'],minutes,health['model'])
        EXTERNAL['manifest']=manifest
        generation=secrets.token_hex(12);EXTERNAL['generation']=generation
        timer=threading.Timer(minutes*60,lambda:expire_external(generation,stop_server_at_expiry))
        timer.daemon=True;EXTERNAL['timer']=timer;timer.start()
    except Exception:
        stop_external(False)
        raise RuntimeError('外部入口未建立。检查ngrok账户/网络/配额；未输出敏感异常详情。可重新执行启动。') from None
    print('外部 base_url:',manifest['base_url'])
    print('model:',manifest['model'],'；认证读取本地环境变量 STRATA_API_KEY；到期:',manifest['expires_at'])
    print('SSE已启用；心跳仅用于当前请求的网络传输，不延长Colab会话。')
    return manifest


def stop_external(stop_server=False):
    with EXTERNAL_LOCK:return _stop_external(stop_server)


def start_external(minutes=60,body_mib=8,concurrency=1,stream_timeout=3600,stop_server_at_expiry=True):
    with EXTERNAL_LOCK:
        return _start_external(minutes,body_mib,concurrency,stream_timeout,stop_server_at_expiry)


def expire_external(generation,stop_server):
    with EXTERNAL_LOCK:
        if EXTERNAL.get('generation')==generation:_stop_external(stop_server)


In [ ]:
#@title 11A · 安装外部API依赖（首次需要，默认跳过）
install_external_dependencies = False #@param {type:"boolean"}
if install_external_dependencies:
    run_logged([sys.executable,'-m','pip','install','pyngrok=='+NGROK_VERSION], BASE/'external-install.log')
    print('pyngrok依赖已安装；尚未读取Secrets或创建外部入口')
else:
    print('未安装外部API依赖')


In [ ]:
#@title 12 · 外部API启停（本地Strata先启动并就绪）
external_action = "不操作" #@param ["不操作", "启动外部API", "停止外部API", "停止全部"]
accept_ngrok_traffic_processing = False #@param {type:"boolean"}
confirm_paid_colab_positive_cu = False #@param {type:"boolean"}
external_session_minutes = 60 #@param [15,30,60,120,240] {type:"raw"}
external_body_limit_mib = 8 #@param [1,2,4,8,16] {type:"raw"}
external_concurrency = 1 #@param {type:"integer"}
external_stream_timeout_seconds = 3600 #@param {type:"integer"}
stop_server_at_session_expiry = True #@param {type:"boolean"}
if external_action == '启动外部API':
    if not accept_ngrok_traffic_processing or not confirm_paid_colab_positive_cu:
        raise RuntimeError('先确认ngrok处理流量的隐私影响，以及你的Colab付费/CU状态；不满足时保留内部使用')
    start_external(minutes=external_session_minutes,body_mib=external_body_limit_mib,
        concurrency=external_concurrency,stream_timeout=external_stream_timeout_seconds,
        stop_server_at_expiry=stop_server_at_session_expiry)
elif external_action in ('停止外部API','停止全部'):
    stop_external(stop_server=external_action=='停止全部')
else:
    print('未创建外部入口')


连接约定：
- base_url 使用上个单元实际返回的 https://…ngrok…/v1，model 使用导出会话信息中的实际 model ID
- 所有请求带 Authorization: Bearer（本地环境变量 STRATA_API_KEY），建议带 ngrok-skip-browser-warning: 1
- GET /health、GET /v1/models；POST /v1/chat/completions 支持 stream=false 和 stream=true（SSE）。无其它公开路由，不公开网页/管理接口
- 普通JSON请求最多等240秒；长提示词建议 stream=true。流式会在上游尚未回复前开始SSE，并每15秒发送注释心跳。客户端要忽略“:”注释并检查data中的error；HTTP200本身不代表生成成功
- stream_timeout 与会话到期是实际截止时间，不保证请求一定完成。原模型/Colab也可能超时、OOM或结束；心跳只维持当前请求网络传输，不让Colab永久在线
- 每次新建外部会话都重新导出JSON并更新本地客户端；不要假设旧URL仍有效。同一URL也可能在以后被重用，过期信息必须作废
- 到期默认停止外部入口及本Notebook的Strata服务。**停止模型不会自动断开Colab/GPU计费**，用完仍需在Colab菜单断开/删除运行时


In [ ]:
#@title 13 · 导出无密钥会话信息供本地客户端使用
export_session_info = False #@param {type:"boolean"}
if export_session_info:
    if not EXTERNAL.get('manifest') or not EXTERNAL.get('gateway'):
        raise RuntimeError('先启动外部API；不会导出旧会话')
    SESSION_FILE.write_text(json.dumps(EXTERNAL['manifest'],ensure_ascii=False,indent=2)+'\n')
    SESSION_FILE.chmod(0o600)
    from google.colab import files
    files.download(str(SESSION_FILE))
    print('已准备 strata-session.json，里面没有API key。保留在本地客户端使用，勿当作永久服务地址。')
else:
    print('未导出会话信息')
